# धडा ०५ - एजेंटिक RAG


## सेटअप

हे नोटबुक Microsoft Agent Framework वापरून Agentic RAG (Retrieval-Augmented Generation) पॅटर्नचे प्रदर्शन करते.

**आवश्यकता:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तुमचा Azure AI Search सेवा एंडपॉइंट
- `AZURE_SEARCH_API_KEY` — तुमची Azure AI Search API की
- पर्यावरण चलांद्वारे कॉन्फिगर केलेले Azure OpenAI डिप्लॉयमेंट
- Azure CLI प्रमाणित (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## एजंटिक RAG म्हणजे काय?

पारंपारिक RAG एक निश्चित टप्याच्या प्रक्रियाचे पालन करते: दस्तऐवज प्राप्त करा, नंतर प्रतिक्रिया तयार करा. **एजंटिक RAG** अधिक पुढे जाते ज्याने एजंटला माहिती कधी आणि कशी प्राप्त करायची आहे हे स्वायत्तपणे ठरवण्याचा अधिकार दिला जातो.

एजंटिक RAG सह, एजंट करू शकतो:
- प्रश्नाचे उत्तर देण्याआधी **माहिती प्राप्त करणे आवश्यक आहे का** हे ठरवणे
- कोणत्या डेटा स्त्रोत किंवा साधनाला विचारायचे ते **निवडणे**
- प्राप्त केलेल्या निकालांचे **मूल्यांकन करणे** आणि जर पहिली चाचणी अपुरी असेल तर पुढील माहिती प्राप्त करणे
- एकाधिक पुनर्प्राप्ती टप्प्यांमधून माहिती **संपूर्ण उत्तरात एकत्र करणे**

हे एजंटला स्थिर पुनर्प्राप्ती-नंतर-निर्मिती प्रक्रियेच्या तुलनेत अधिक लवचिक आणि अचूक बनवते.


## शोध साधन तयार करणे

एजेंटिक RAG मध्ये, बाह्य डेटा स्रोतांना **साधने** म्हणून गुंडाळले जाते जे एजंट आवश्यकता भासल्यास वापरु शकतो. हे एजंटला पुनर्प्राप्ती हा आणखी एक क्रिया म्हणून घेण्याची परवानगी देते, ज्यामुळे तो एक अनिवार्य टप्पा नसतो.

खाली आम्ही प्रवास ज्ञानाधार तयार करतो आणि एजंट त्याला कॉल करून गंतव्यस्थानाची माहिती शोधू शकेल अशा प्रकारे ते एक साधन म्हणून उघडतो.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजंट तयार करणे

आता आपण एक असा एजंट तयार करणार आहोत ज्याला **उत्तर देण्यापूर्वी नेहमी माहिती शोधण्याचा आदेश दिला जातो**. एजंट त्याच्या स्वतःच्या प्रशिक्षण डेटावर अवलंबून न राहता ज्ञानाधारित उत्तरांसाठी `search_travel_knowledge` साधन वापरतो.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्ती मिळवणे — मेकर-चेकर नमुना

एजंटिक RAG चे एक मुख्य फायदे म्हणजे **पुनरावृत्ती मिळवणे**. एजंट अनेकदा शोध घेऊ शकतो ज्यामुळे त्याच्या प्राथमिक निष्कर्षांची तपासणी, सुधारणा किंवा विस्तार करता येतो — ज्यामुळे "मेकर-चेकर" कार्यप्रवाह सारखा अनुभव येतो:

1. **मेकर पायरी**: एजंट प्राथमिक माहिती प्राप्त करतो आणि उत्तर तयार करतो.
2. **चेकर पायरी**: एजंट तपशीलांची पुष्टी करण्यासाठी किंवा रिकाम्या जागा भरून काढण्यासाठी अतिरिक्त शोध घेतो.

खाली, एजंटला असा प्रश्न विचारला आहे ज्यासाठी अनेक गंतव्यांची तुलना करणे आवश्यक आहे, ज्यामुळे त्याला अनेक वेळा शोध घेण्यास भाग पडते.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework वापरून **एजेन्टिक RAG** प्रणाली कशी तयार करायची हे शिकलात:

- **एजेन्टिक RAG** एजंटांना माहिती पुनर्प्राप्ती कधी करायची ते स्वयंचलितपणे ठरवण्याची परवानगी देते, ज्यामुळे पुनर्प्राप्ती निश्चित न राहता गतिशील होते.
- **टूल्स म्हणून डेटा स्रोत**: बाह्य ज्ञान आधार (उदा. Azure AI Search) हे एजंट वापरू शकणाऱ्या टूल्स म्हणून सज्ज केलेले असतात.
- **परावर्तनीय पुनर्प्राप्ती**: मेकर-चेककर नमुना एजंटला अनेक पुनर्प्राप्ती फेरी पार पाडण्यास सक्षम करतो — शोध घेणे, पडताळणी करणे, आणि परिष्कृत करणे — अंतिम उत्तर तयार करण्यापूर्वी.

उत्पादनात, तुम्ही मोठ्या प्रमाणात प्रवास दस्तऐवज पुनर्प्राप्तीसाठी रिअल Azure AI Search निर्देशांकाने `TRAVEL_KNOWLEDGE_BASE` चा वापर बदलाल.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
